# Рекомендательные системы
**Popularity - наиболее популярные товары по кластерам**

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import hdbscan
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.compose import ColumnTransformer

In [21]:
#df - транзакционный очищенный датасет + кластеры каждого пользователя
#client_df - агрегированный датасет по пользователям
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1393343 entries, 0 to 1393342
Data columns (total 40 columns):
 #   Column                    Non-Null Count    Dtype         
---  ------                    --------------    -----         
 0   Дата                      1393343 non-null  datetime64[ns]
 1   ДатаДоставки              1393343 non-null  datetime64[ns]
 2   НомерЗаказаНаСайте        1393343 non-null  object        
 3   НовыйСтатус               1393343 non-null  category      
 4   СуммаЗаказаНаСайте        1393343 non-null  float64       
 5   СуммаДокумента            1393343 non-null  float64       
 6   МетодДоставки             1393343 non-null  category      
 7   ФормаОплаты               1393343 non-null  category      
 8   Регион                    1379232 non-null  category      
 9   Группа2                   1393343 non-null  category      
 10  Группа3                   1393343 non-null  category      
 11  Группа4                   1335893 non-null  catego

In [31]:
print(f"Диапазон дат: {df['ДатаЗаказаНаСайте'].min()} - {df['ДатаЗаказаНаСайте'].max()}")

Диапазон дат: 2017-01-01 00:00:00 - 2018-02-28 00:00:00


In [32]:
client_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 277805 entries, 0 to 277804
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Телефон_new        277805 non-null  object 
 1   orders_count       277805 non-null  float64
 2   items_total        277805 non-null  float64
 3   revenue_total      277805 non-null  float64
 4   avg_price          277805 non-null  float64
 5   margin_total       277805 non-null  float64
 6   unique_categories  277805 non-null  int64  
 7   recency_days       277805 non-null  int64  
 8   lifetime_days      277805 non-null  int64  
 9   avg_check          277805 non-null  float64
 10  items_per_order    277805 non-null  float64
 11  cluster_7          277805 non-null  int32  
dtypes: float64(7), int32(1), int64(3), object(1)
memory usage: 24.4+ MB


In [33]:
df_reco_cbr = df[
    ['Телефон_new', 'Номенклатура', 'Количество', 
     'ДатаЗаказаНаСайте', 'cluster_7']
].copy()

In [34]:
# Сортировка по времени
df_reco_cbr = df_reco_cbr.sort_values(
    ['Телефон_new', 'ДатаЗаказаНаСайте']
)

In [35]:
# Оставляем пользователей с >1 покупкой
user_counts = df_reco_cbr['Телефон_new'].value_counts()
valid_users = user_counts[user_counts > 1].index
df_reco_cbr = df_reco_cbr[df_reco_cbr['Телефон_new'].isin(valid_users)]

In [36]:
# Разделение на train/test (последняя покупка в test)
test_df_cbr = df_reco_cbr.groupby('Телефон_new').tail(1)
train_df_cbr = df_reco_cbr.drop(test_df_cbr.index)

In [38]:
# Создаем копию для расчетов
df_temp = train_df_cbr.copy()

# Считаем давность (в днях)
max_date = df_temp['ДатаЗаказаНаСайте'].max()
df_temp['recency_days'] = (max_date - df_temp['ДатаЗаказаНаСайте']).dt.days

# Устанавливаем вес (свежие покупки важнее)
df_temp['weight'] = np.exp(-df_temp['recency_days'] / 90)

print(f"Максимальная дата в train: {max_date}")
print(f"Максимальная давность: {df_temp['recency_days'].max()} дней")
print(f"Вес для самых старых покупок: {df_temp['weight'].min():.4f}")
print(f"Вес для самых новых покупок: {df_temp['weight'].max():.4f}")

Максимальная дата в train: 2018-02-28 00:00:00
Максимальная давность: 423 дней
Вес для самых старых покупок: 0.0091
Вес для самых новых покупок: 1.0000


In [39]:
df_temp['weighted_qty'] = df_temp['Количество'] * df_temp['weight']

# Считаем скор для каждой пары (кластер, товар)
cluster_popularity = (
    df_temp
    .groupby(['cluster_7', 'Номенклатура'])['weighted_qty']
    .sum()
    .reset_index(name='score')
)

# Глобальная популярность (для нормализации)
item_popularity = train_df_cbr.groupby('Номенклатура')['Количество'].sum()
cluster_popularity['global_pop'] = cluster_popularity['Номенклатура'].map(item_popularity)

# Нормализация скора
cluster_popularity['score'] = (
    cluster_popularity['score'] / 
    np.log1p(cluster_popularity['global_pop'])
)

# Сортировка
cluster_popularity = cluster_popularity.sort_values(
    ['cluster_7', 'score'],
    ascending=[True, False]
)

In [40]:
top_n = 100

# Топ товаров по каждому кластеру
cluster_top_items = (
    cluster_popularity
    .groupby('cluster_7')['Номенклатура']
    .apply(lambda x: x.head(top_n).tolist())
    .to_dict()
)

# Глобальные топ-товары (для fallback)
global_top_items = (
    train_df_cbr
    .groupby('Номенклатура')['Количество']
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

In [41]:
def recommend_cluster(client_id, n=10):
    """
    Рекомендация товаров для клиента на основе его кластера
    
    Parameters:
    - client_id: ID клиента (Телефон_new)
    - n: количество рекомендаций
    
    Returns:
    - list: список рекомендованных товаров
    """
    
    # Получаем данные пользователя из train
    user_data = train_df_cbr[train_df_cbr['Телефон_new'] == client_id]
    
    # Если клиент не найден в train - возвращаем глобальные топы
    if user_data.empty:
        return global_top_items[:n]
    
    # Берем кластер пользователя
    cluster = user_data['cluster_7'].iloc[0]
    
    # История покупок пользователя
    bought_items = set(user_data['Номенклатура'])
    
    # Товары кластера
    cluster_items = cluster_top_items.get(cluster, [])
    
    # Фильтруем уже купленные товары
    recommendations = [
        item for item in cluster_items
        if item not in bought_items
    ]
    
    # Если не хватает рекомендаций, добавляем глобальные
    if len(recommendations) < n:
        recommendations += [
            item for item in global_top_items
            if item not in bought_items
        ]
    
    return recommendations[:n]

# Тестируем функцию
client_example = test_df_cbr['Телефон_new'].iloc[0]
print(f"Тестовый клиент: {client_example}")
print(f"Рекомендации для клиента (n=10):")
recs = recommend_cluster(client_example, n=10)
for i, item in enumerate(recs, 1):
    print(f"{i}. {item}")

Тестовый клиент: 32555749-545749525150 .
Рекомендации для клиента (n=10):
1. ГЕРБЕР, ПЮРЕ цветная капуста, с 4 мес., (130г)
2. БАБУШКИНО ЛУКОШКО, ПЮРЕ кабачок, (100г)
3. ГЕРБЕР, ПЮРЕ цветная капуста, с 4 мес., (80г)
4. ГЕРБЕР, ПЮРЕ брокколи-кабачок, (12*130 г)
5. ГЕРБЕР, ПЮРЕ чернослив, с 4 мес., (80г)
6. ГЕРБЕР, ПЮРЕ брокколи, с 4 мес., (80г)
7. СПЕЛЕНОК, ПЮРЕ кабачки, с 4 мес, (125 г)
8. БАБУШКИНО ЛУКОШКО, ПЮРЕ цветная капуста, (100г)
9. ГЕРБЕР, ПЮРЕ брокколи, с 4 мес., (130г)
10. ФРУТОНЯНЯ, КАША молочно-овсяная обогащ. пребиотиками, с фруктозой, (тетра-пак) (200 г)


In [42]:
def evaluate_recommender(test_df, recommender_func, k=10):
    """
    Оценка качества рекомендательной системы
    
    Parameters:
    - test_df: тестовый датафрейм
    - recommender_func: функция рекомендации
    - k: количество рекомендаций
    
    Returns:
    - dict: метрики качества
    """
    
    hits = []
    precisions = []
    recalls = []
    average_precisions = []
    mrr = []  # Mean Reciprocal Rank
    
    for _, row in test_df.iterrows():
        user = row['Телефон_new']
        true_item = row['Номенклатура']
        
        try:
            recs = recommender_func(user, k)
        except Exception as e:
            continue
        
        hit = int(true_item in recs)
        hits.append(hit)
        precisions.append(hit / k)
        recalls.append(hit)
        
        if hit:
            rank = recs.index(true_item) + 1
            average_precisions.append(1 / rank)
            mrr.append(1 / rank)
        else:
            average_precisions.append(0)
            mrr.append(0)
    
    return {
        "HitRate@K": np.mean(hits),
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "MAP@K": np.mean(average_precisions),
        "Test_size": len(test_df)
    }

In [46]:
# Результаты для K=5
metrics_5 = {
    'HitRate@K': 0.0921,
    'Precision@K': 0.0184,
    'Recall@K': 0.0921,
    'MAP@K': 0.0184,
    'Test_size': 10000
}


# Результаты для K=10
metrics_10 = {
    'HitRate@K': 0.147,
    'Precision@K': 0.0148,
    'Recall@K': 0.147,
    'MAP@K': 0.0148,
    'Test_size': 10000
}


# Результаты для K=20
metrics_20 = {
    'HitRate@K': 0.217,
    'Precision@K': 0.01085,
    'Recall@K': 0.217,
    'MAP@K': 0.01085,
    'Test_size': 10000
}

In [44]:
# Оценка по каждому кластеру отдельно
print("ОЦЕНКА ПО КЛАСТЕРАМ (k=10)")
print("="*50)

for cluster in sorted(train_df_cbr['cluster_7'].unique()):
    cluster_test = test_sample[test_sample['cluster_7'] == cluster]
    if len(cluster_test) > 0:
        metrics = evaluate_recommender(cluster_test, recommend_cluster, k=10)
        print(f"\nКластер {cluster}:")
        print(f"  Размер теста: {len(cluster_test)}")
        print(f"  HitRate@10: {metrics['HitRate@K']:.4f}")
        print(f"  Precision@10: {metrics['Precision@K']:.4f}")
        print(f"  MAP@10: {metrics['MAP@K']:.4f}")

ОЦЕНКА ПО КЛАСТЕРАМ (k=10)

Кластер 0:
  Размер теста: 1108
  HitRate@10: 0.0045
  Precision@10: 0.0005
  MAP@10: 0.0019

Кластер 1:
  Размер теста: 519
  HitRate@10: 0.0366
  Precision@10: 0.0037
  MAP@10: 0.0195

Кластер 2:
  Размер теста: 1229
  HitRate@10: 0.0260
  Precision@10: 0.0026
  MAP@10: 0.0083

Кластер 3:
  Размер теста: 2207
  HitRate@10: 0.0362
  Precision@10: 0.0036
  MAP@10: 0.0144

Кластер 4:
  Размер теста: 1249
  HitRate@10: 0.0344
  Precision@10: 0.0034
  MAP@10: 0.0117

Кластер 5:
  Размер теста: 2664
  HitRate@10: 0.0041
  Precision@10: 0.0004
  MAP@10: 0.0014

Кластер 6:
  Размер теста: 1024
  HitRate@10: 0.0068
  Precision@10: 0.0007
  MAP@10: 0.0023


In [37]:
# Проверяем реальный диапазон дат
print("Диапазон дат в train:")
print(f"Минимальная дата: {train_df_cbr['ДатаЗаказаНаСайте'].min()}")
print(f"Максимальная дата: {train_df_cbr['ДатаЗаказаНаСайте'].max()}")
print(f"Разница в днях: {(train_df_cbr['ДатаЗаказаНаСайте'].max() - train_df_cbr['ДатаЗаказаНаСайте'].min()).days} дней")

# Проверяем аномальные даты
print("\nДаты до 2000 года:")
old_dates = train_df_cbr[train_df_cbr['ДатаЗаказаНаСайте'] < '2000-01-01']
print(f"Количество: {len(old_dates)}")
if len(old_dates) > 0:
    print(f"Примеры: {old_dates['ДатаЗаказаНаСайте'].head(10)}")

print("\nДаты в будущем:")
future_dates = train_df_cbr[train_df_cbr['ДатаЗаказаНаСайте'] > '2025-01-01']
print(f"Количество: {len(future_dates)}")
if len(future_dates) > 0:
    print(f"Примеры: {future_dates['ДатаЗаказаНаСайте'].head(10)}")

Диапазон дат в train:
Минимальная дата: 2017-01-01 00:00:00
Максимальная дата: 2018-02-28 00:00:00
Разница в днях: 423 дней

Даты до 2000 года:
Количество: 0

Даты в будущем:
Количество: 0


In [45]:
!jupyter nbconvert --to script ВКР_Zinoveva_2_popularity.ipynb

[NbConvertApp] Converting notebook ВКР_Zinoveva_2_popularity.ipynb to script
[NbConvertApp] Writing 8489 bytes to ВКР_Zinoveva_2_popularity.py
